# Active-Set Visualization

This notebook focuses on the active-set story behind the tiny LP. The main question is not whether the solver gets close to the right objective, but whether its returned sample is close enough to a true vertex for active-set logic and warm-start reconstruction to work reliably.


In [2]:
from __future__ import annotations

import numpy as np
from IPython.display import Markdown, display

np.set_printoptions(suppress=True, precision=6)


def fmt(value, digits: int = 6) -> str:
    if value is None:
        return 'None'
    if isinstance(value, (np.bool_, bool)):
        return 'True' if bool(value) else 'False'
    if isinstance(value, (int, np.integer)):
        return str(int(value))
    if isinstance(value, (float, np.floating)):
        return f'{float(value):.{digits}f}'
    if isinstance(value, str):
        return value
    if isinstance(value, tuple):
        return '[' + ', '.join(fmt(v, digits=digits) for v in value) + ']'
    if isinstance(value, (list, np.ndarray)):
        arr = np.asarray(value)
        if arr.ndim == 0:
            return fmt(arr.item(), digits=digits)
        return '[' + ', '.join(fmt(v, digits=digits) for v in arr.tolist()) + ']'
    return str(value)


def markdown_table(headers, rows):
    lines = [
        '| ' + ' | '.join(headers) + ' |',
        '| ' + ' | '.join(['---'] * len(headers)) + ' |',
    ]
    for row in rows:
        lines.append('| ' + ' | '.join(str(cell) for cell in row) + ' |')
    return '\n'.join(lines)


def show_table(headers, rows, title: str | None = None):
    parts = []
    if title:
        parts.append(f'### {title}')
    parts.append(markdown_table(headers, rows))
    display(Markdown('\n\n'.join(parts)))


def show_bullets(title: str, items):
    display(Markdown('### ' + title + '\n' + '\n'.join(f'- {item}' for item in items)))


def mask_to_bits(mask) -> str:
    return ''.join('1' if bool(v) else '0' for v in mask)

from collections import Counter

from lpas.core.active_set import primal_active_mask
from lpas.experiments.generators import tiny_known_lp
from lpas.solver.adaptive_solver import AdaptiveLPSolver
from lpas.solver.scipy_handoff import solve_with_scipy
from lpas.solver.warm_start import reconstruct_from_active_set
from lpas.utils.config import SamplerConfig, SolverConfig


## Set Up the Same Solver Run

Using the same seed and configuration as the basic demo makes the interpretation consistent across notebooks.


In [3]:
problem = tiny_known_lp()
config = SolverConfig(
    batch_size=512,
    max_iter=80,
    patience=20,
    sampler=SamplerConfig(
        seed=42,
        sigma_init=2.5,
        primal_init_mean=0.75,
        dual_init_mean=0.75,
    ),
)
result = AdaptiveLPSolver(config).solve(problem)
scipy = solve_with_scipy(problem)


TypeError: ScipySolveResult.__init__() got an unexpected keyword argument 'nonneg_active_mask'. Did you mean 'primal_active_mask'?

## Slack at the Returned Sample

A constraint is only considered active when its slack falls below the configured tolerance. That means a point can be visually near a boundary and still look inactive to the discrete active-set extractor.


In [ ]:
constraint_labels = ['`x1 + x2 <= 4`', '`x1 <= 2`', '`x2 <= 3`']
adaptive_slack = problem.b - problem.A @ result.best_x
scipy_slack = problem.b - problem.A @ scipy.x
slack_rows = [
    (label, fmt(a), fmt(s), fmt(a - s))
    for label, a, s in zip(constraint_labels, adaptive_slack, scipy_slack)
]
show_table(
    ['Constraint', 'Adaptive slack', 'SciPy slack', 'Adaptive - SciPy'],
    slack_rows,
    title='Distance from each boundary',
)


## Tolerance Sweep

Here I vary the activity tolerance `epsilon` and compare the primal mask for the adaptive point against the exact SciPy optimum. The bit ordering is `[x1 + x2 <= 4, x1 <= 2, x2 <= 3]`.


In [ ]:
tolerances = [1e-6, 1e-4, 1e-3, 1e-2]
rows = []
for epsilon in tolerances:
    adaptive_mask = primal_active_mask(problem, result.best_x, epsilon=epsilon)
    scipy_mask = primal_active_mask(problem, scipy.x, epsilon=epsilon)
    hint = reconstruct_from_active_set(problem, adaptive_mask)
    rows.append(
        (
            f'{epsilon:.0e}',
            mask_to_bits(adaptive_mask),
            mask_to_bits(scipy_mask),
            hint.message,
            'None' if hint.objective is None else fmt(hint.objective),
        )
    )
show_table(
    ['epsilon', 'Adaptive mask', 'SciPy mask', 'Warm-start outcome', 'Recovered objective'],
    rows,
    title='Sensitivity of active-set extraction',
)


## Dominant Archive Patterns Over Time

The solver history stores the most common active-set pattern in the archive after each iteration. Even without a plot, the frequency table shows whether the search is repeatedly returning vertex-like samples or mostly staying in the interior.


In [ ]:
pattern_counts = Counter(entry.dominant_active_pattern for entry in result.history)
pattern_rows = []
for pattern, count in pattern_counts.most_common():
    primal_pattern, dual_pattern = pattern
    pattern_rows.append(
        (
            mask_to_bits(primal_pattern),
            mask_to_bits(dual_pattern),
            count,
            f'{100.0 * count / len(result.history):.1f}%',
        )
    )
show_table(
    ['Primal bits', 'Dual bits', 'Iterations', 'Share of history'],
    pattern_rows,
    title='Dominant pattern frequencies',
)


In [ ]:
empty_pattern_share = 100.0 * pattern_counts.get(((False, False, False), (False, False)), 0) / len(result.history)
insights = [
    f'The adaptive candidate is still `{adaptive_slack[0]:.6f}` and `{adaptive_slack[1]:.6f}` away from the two truly binding constraints, so the default `1e-6` activity threshold reports an empty primal mask.',
    'At `epsilon = 1e-2`, the adaptive mask becomes `110`, which matches the SciPy vertex and is sufficient to reconstruct the exact solution `[2.0, 2.0]` with objective `10.0`.',
    f'The dominant archive pattern is the fully inactive primal mask for the entire run, and the completely inactive primal-dual pair `000 / 00` accounts for `{empty_pattern_share:.1f}%` of history entries.',
    'This experiment shows a real limitation of hard active-set thresholds: approximate samples can contain the right geometry while still producing the wrong discrete support pattern.',
]
show_bullets('Insights', insights)
